In [1]:
!pip install -q transformers torch

In [2]:
import json
import random
import torch

# HuggingFace transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer

## Loading our saved subsets from Drive

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load datasets from your genai folder
with open("/content/drive/MyDrive/genai/eval_subset.json", "r") as f:
    eval_subset = json.load(f)

with open("/content/drive/MyDrive/genai/train_subset.json", "r") as f:
    train_subset = json.load(f)

print(f"Eval examples: {len(eval_subset)}")
print(f"Train examples: {len(train_subset)}")

Mounted at /content/drive
Eval examples: 500
Train examples: 5000


## Loading GPT-2

In [4]:
# Load GPT-2 small model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set padding token (important for generation)
tokenizer.pad_token = tokenizer.eos_token

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Model loaded on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on cuda


Prompt builder

In [5]:
def build_prompt(serialized_table, examples=None):
    """
    Build a prompt for GPT-2.

    If examples are provided → few-shot
    Otherwise → zero-shot
    """

    prompt = ""

    # few-shot examples
    if examples:
        for ex in examples:
            prompt += f"Table: {ex['serialized_table']}\n"
            prompt += f"Description: {ex['reference']}\n\n"

    # target table
    prompt += f"Table: {serialized_table}\n"
    prompt += "Description:"

    return prompt

Generation Function

In [6]:
def generate_text(prompt, max_new_tokens_val=200):
    """
    Generate text using GPT-2 given a prompt
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens_val,
        do_sample=True,        # enables randomness
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated[len(prompt):].strip()

Zero-shot example

In [7]:
example = eval_subset[0]

prompt = build_prompt(example["serialized_table"])

output = generate_text(prompt, max_new_tokens_val=200)

print("=== ZERO-SHOT ===")
print("Table:", example["serialized_table"])
print("\nGenerated:", output)
print("\nReference:", example["reference"])

=== ZERO-SHOT ===
Table: # : 76 | Took Office : Daniel Henry Chamberlain | Left Office : December 1, 1874

Generated: The first office to be occupied by Chamberlain, and a first in modern history, was occupied by two men on January 27, 1874, by the wife of his deputy, and by a second on February 1.
The first office was occupied by Henry Chamberlain, a man of military ability. He took office on July 5, 1874, and in August 1875, during the Great War, was appointed as the head of the Government. He resigned on September 5, 1874, but resigned on October 1, 1875, and became a member of the Senate and Governor-General of England, the position of the second office.
He became a member of the House of Commons, in 1878. In 1883 he was elected in the House of Lords, and became a member of the House of Lords Council.
He was elected as a member of the House of Lords in 1887, and as a member of the Commons in 1889. He was elected to the House of Commons

Reference: Daniel Henry Chamberlain was the 7

Few-shot

In [8]:
# Selecting 3 random training examples
few_shot_examples = random.sample(train_subset, 3)

example = eval_subset[1]

prompt = build_prompt(example["serialized_table"], few_shot_examples)

output = generate_text(prompt, max_new_tokens_val=200)

print("=== FEW-SHOT (3) ===")
print("Generated:", output)

=== FEW-SHOT (3) ===
Generated: Evelyn was a dancer.

Table: Gender : Blonde | Role : Evelyn

Description: Evelyn was a woman.

Table: Year : 2016 | Title : Children of Evelyn | Role : Evelyn

Description: Evelyn was a woman.

Table: Year : 2016 | Title : Children of Evelyn | Role : Evelyn

Description: Evelyn was a woman.

Table: Year : 2016 | Title : Children of Evelyn | Role : Evelyn

Description: Evelyn was a woman.

Table: Year : 2016 | Title : Children of Evelyn | Role : Evelyn

Description: Evelyn was a woman.

Table: Year : 2016 | Title : Children of Evelyn | Role : Evelyn

Description: Evelyn was a woman.

Table: Year : 2016 | Title : Children of Evelyn | Role : Evelyn

Description: Evelyn was


In [9]:
for k in [5, 10]:
    few_shot_examples = random.sample(train_subset, k)

    example = eval_subset[2]
    prompt = build_prompt(example["serialized_table"], few_shot_examples)

    output = generate_text(prompt, max_new_tokens_val=200)

    print(f"\n=== FEW-SHOT ({k}) ===")
    print("Generated:", output)


=== FEW-SHOT (5) ===
Generated: Total: 119

Table: event : 800 m | Athlete : Peter Snell | Nationality : 20 July 2018

Description: Peter Snell was the Oceanian record holder in the 800 m until 20 July 2018.

Table: TOTAL : 121

Description: Total: 121

Table: event : 800 m | Athlete : Peter Snell | Nationality : 20 July 2018

Description: Peter Snell was the Oceanian record holder in the 800 m until 20 July 2018.

Table: total : 121

Description: total : 121

Table: event : 800 m | Athlete : Peter Snell | Nationality : 20 July 2018

Description: Peter Snell was the Oceanian record holder in the 800 m until 20 July 2018.

Table: total : 121

Description: total : 121

Table: event : 800 m | Athlete : Peter Snell | Nationality : 20 July 2018

=== FEW-SHOT (10) ===
Generated: A total of 119 plays were played in the Total and Total 2Q12 games.

Table: Total: 12.57 | T1x : 1:2 | Total: 12.57 | Total: 12.57 | Total: 11.33 | Total: 11.33 | Total: 9.31 | Total: 7.89 | Total: 6.89 | Total: 4.6

In [10]:
results = []

for i in range(10):  # small test first
    ex = eval_subset[i]

    prompt = build_prompt(ex["serialized_table"])
    output = generate_text(prompt, max_new_tokens_val=200)

    results.append({
        "table": ex["serialized_table"],
        "reference": ex["reference"],
        "generated": output
    })

print("Generated 10 examples")

Generated 10 examples


In [11]:
for i in range(len(results)):
    print(f"\n--- Example {i+1} ---")
    print("TABLE:\n", results[i]["table"][:200], "...")
    print("\nREF:", results[i]["reference"])
    print("\nGEN:", results[i]["generated"])


--- Example 1 ---
TABLE:
 # : 76 | Took Office : Daniel Henry Chamberlain | Left Office : December 1, 1874 ...

REF: Daniel Henry Chamberlain was the 76th Governor of South Carolina from 1874.

GEN: This work is of the Royal Society of the British Empire. It was published in 1776. The title is of the British Empire, and the title is a reference to the monarchy. There is a line that reads:
The British Empire
This work is of the Royal Society of the British Empire. It was published in 1776. The title is of the British Empire, and the title is a reference to the monarchy. There is a line that reads:
A letter to the King
"The Royal Society of the British Empire is hereby pleased to express our wish to express our deep gratitude to the following men and women who have received our most recent letter of thanks, which is written in the most friendly spirit and in the most kindly tone." —Letter to Daniel Henry Chamberlain | Left Office : January 3, 1877
Description:
This work is of the Royal 

In [12]:
output_path = "/content/drive/MyDrive/genai/gpt2_results.json"
with open(output_path, "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved results to {output_path}")

Saved results to /content/drive/MyDrive/genai/gpt2_results.json
